# 2.5 Auto-Compiling Python → C++ Translation Agent

The goal of this section is to build an agentic loop to translate **python** code into standalone **C++17** code, compiles and executes the generated code, and uses a feedback mechanism to use compiler/runtime errors to iteratively correct failed translations.


### 1. Check and Configure GPU

In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


### 2. Install Required Dependencies

In [2]:
!pip install -q -U transformers accelerate
!g++ --version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 70.2 MB/s eta 0:00:00
g++ (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



### 3. Load Qwen and Tokenizer

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded


### 4. Model Generation Function and Initial Tests
Create function `generate_response` to input prompt and output response of model

In [4]:
def generate_response(messages, max_new_tokens=1024):

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        [text],
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    generated_ids = outputs[0][inputs.input_ids.shape[-1]:]

    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    return response

Basic Hello World generation test

In [5]:
messages = [
    {
        "role": "user",
        "content": "Write a C++17 program that prints Hello World."
    }
]

response = generate_response(messages)

print(response)

Certainly! Below is a simple C++17 program that prints "Hello World" to the console:

```cpp
#include <iostream>

int main() {
    std::cout << "Hello World!" << std::endl;
    return 0;
}
```

### Explanation:
- `#include <iostream>`: This line includes the input-output stream library, which is necessary for using `std::cout`.
- `int main()`: This is the entry point of the program. The `main` function returns an integer value, which is typically 0 to indicate successful execution.
- `std::cout << "Hello World!" << std::endl;`: This line outputs the string "Hello World!" to the console followed by a newline character (`\n`).
- `return 0;`: This statement indicates that the program has executed successfully.

To compile and run this program, you can use a C++ compiler like `g++`. Here's how you can do it:

1. Save the code in a file named `hello_world.cpp`.
2. Open a terminal or command prompt.
3. Navigate to the directory where the file is saved.
4. Compile the program using the follow

Merge sort python code for test input

In [6]:
sample_python = """
def merge_sort(arr):
    if len(arr) <= 1:
        return arr

    mid = len(arr) // 2

    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])

    result = []
    i = 0
    j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1

    result.extend(left[i:])
    result.extend(right[j:])

    return result

print(merge_sort([7, 2, 9, 1, 5, 3]))
"""

Prompt template with task description and python code

In [7]:
prompt = f"""
Translate the following Python code into standalone C++17.

Requirements:
- Preserve the behavior of the Python code.
- Produce complete compilable C++17 code.
- Include all required headers.
- Include a main() function.
- Do not use Python or pseudocode.

Python code:

{sample_python}

Return only the C++ code.
"""

Initial python → C++ translation test with merge sort

In [8]:
messages = [
    {
        "role": "system",
        "content": "You are an expert Python-to-C++ translator."
    },
    {
        "role": "user",
        "content": prompt
    }
]

response = generate_response(messages)

print(response)

Certainly! Below is the complete C++17 code that translates the given Python `merge_sort` function into standalone C++ code. This code includes all necessary headers, a `main()` function, and compiles correctly.

```cpp
#include <iostream>
#include <vector>

// Function to perform merge sort
std::vector<int> mergeSort(const std::vector<int>& arr) {
    if (arr.size() <= 1) {
        return arr;
    }

    int mid = arr.size() / 2;

    // Recursively sort the left and right halves
    std::vector<int> left = mergeSort(std::vector<int>(arr.begin(), arr.begin() + mid));
    std::vector<int> right = mergeSort(std::vector<int>(arr.begin() + mid, arr.end()));

    std::vector<int> result;
    size_t i = 0, j = 0;

    // Merge the sorted halves
    while (i < left.size() && j < right.size()) {
        if (left[i] <= right[j]) {
            result.push_back(left[i]);
            ++i;
        } else {
            result.push_back(right[j]);
            ++j;
        }
    }

    // Append rema

### 5. Parse and Extract C++ Code
Create function `extract_cpp` that uses `regex` to parse and extract C++ code from model response

In [9]:
import re

def extract_cpp(response):
    match = re.search(r"```(?:cpp|c\+\+)?\s*(.*?)```", response, re.DOTALL)

    if match:
        return match.group(1).strip()

    return response.strip()

### 6. Compile and Execute Generated C++
Create function `compile_and_run` to run and record parsed code from response

In [10]:
import subprocess
import tempfile
import os

def compile_and_run(cpp_code):
    with tempfile.TemporaryDirectory() as tmpdir:
        cpp_file = os.path.join(tmpdir, "program.cpp")
        exe_file = os.path.join(tmpdir, "program")

        with open(cpp_file, "w") as f:
            f.write(cpp_code)

        compile_result = subprocess.run(
            ["g++", "-std=c++17", cpp_file, "-o", exe_file],
            capture_output=True,
            text=True
        )

        if compile_result.returncode != 0:
            return False, compile_result.stderr

        run_result = subprocess.run(
            [exe_file],
            capture_output=True,
            text=True,
            timeout=10
        )

        if run_result.returncode != 0:
            return False, run_result.stderr

        return True, run_result.stdout

Test constructed pipeline on merge sort test input

In [11]:
cpp_code = extract_cpp(response)

print("Extracted C++:")
print(cpp_code)

success, output = compile_and_run(cpp_code)

print("\nSuccess:", success)
print("Output:", output)

Extracted C++:
#include <iostream>
#include <vector>

// Function to perform merge sort
std::vector<int> mergeSort(const std::vector<int>& arr) {
    if (arr.size() <= 1) {
        return arr;
    }

    int mid = arr.size() / 2;

    // Recursively sort the left and right halves
    std::vector<int> left = mergeSort(std::vector<int>(arr.begin(), arr.begin() + mid));
    std::vector<int> right = mergeSort(std::vector<int>(arr.begin() + mid, arr.end()));

    std::vector<int> result;
    size_t i = 0, j = 0;

    // Merge the sorted halves
    while (i < left.size() && j < right.size()) {
        if (left[i] <= right[j]) {
            result.push_back(left[i]);
            ++i;
        } else {
            result.push_back(right[j]);
            ++j;
        }
    }

    // Append remaining elements from left half
    result.insert(result.end(), left.begin() + i, left.end());

    // Append remaining elements from right half
    result.insert(result.end(), right.begin() + j, right.end()

### 7. Construct Feedback Fix Loop
Create function `translate_and_verify` to translate and verify python to C++17 and iteratively correct failed translation based error feedback.

In [12]:
def translate_and_verify(python_code, max_retries=3):

    # Initial translation prompt
    prompt = f"""
Translate the following Python code into standalone C++17.

Requirements:
- Preserve the behavior of the Python code.
- Produce complete compilable C++17 code.
- Include all required headers.
- Include a main() function.
- Return only the C++ code.

Python code:

{python_code}
"""

    messages = [
        {
            "role": "system",
            "content": "You are an expert Python-to-C++ translator."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    # Try up to max_retries times
    for attempt in range(max_retries):

        print(f"\nAttempt {attempt + 1}/{max_retries}")

        # 1. Ask Qwen for C++
        response = generate_response(messages)

        # 2. Extract clean C++
        cpp_code = extract_cpp(response)

        # 3. Compile and execute it
        success, result = compile_and_run(cpp_code)

        # 4. If it works, we're done
        if success:
            print("✓ Compilation and execution successful!")
            return cpp_code

        # 5. If it failed, show the error
        print("✗ Verification failed.")
        print(result)

        # 6. Give the failed code + error back to Qwen
        fix_prompt = make_fix_prompt(
            python_code,
            cpp_code,
            result
        )

        messages = [
            {
                "role": "system",
                "content": "You are an expert Python-to-C++ translator."
            },
            {
                "role": "user",
                "content": fix_prompt
            }
        ]

    # All attempts failed
    raise RuntimeError(
        f"C++ translation failed after {max_retries} attempts."
    )

Test current pipeline on merge sort input

In [13]:
cpp_result = translate_and_verify(sample_python, max_retries=3)

if cpp_result is not None:
    print("\nFinal C++:")
    print(cpp_result)


Attempt 1/3
✓ Compilation and execution successful!

Final C++:
#include <iostream>
#include <vector>

std::vector<int> merge_sort(const std::vector<int>& arr) {
    if (arr.size() <= 1) {
        return arr;
    }

    size_t mid = arr.size() / 2;

    std::vector<int> left = merge_sort(std::vector<int>(arr.begin(), arr.begin() + mid));
    std::vector<int> right = merge_sort(std::vector<int>(arr.begin() + mid, arr.end()));

    std::vector<int> result;
    size_t i = 0, j = 0;

    while (i < left.size() && j < right.size()) {
        if (left[i] <= right[j]) {
            result.push_back(left[i]);
            ++i;
        } else {
            result.push_back(right[j]);
            ++j;
        }
    }

    result.insert(result.end(), left.begin() + i, left.end());
    result.insert(result.end(), right.begin() + j, right.end());

    return result;
}

int main() {
    std::vector<int> arr = {7, 2, 9, 1, 5, 3};
    std::vector<int> sorted_arr = merge_sort(arr);

    std::cout << "S

### 8. Error Feedback and Self-Correction
Create function `make_fix_prompt` to construct prompt from input, failed response and error

In [14]:
def make_fix_prompt(python_code, cpp_code, error):

    return f"""
You are an expert Python-to-C++17 translator.

The following Python code must be translated into standalone C++17.

Python code:
{python_code}

Your previous C++ translation was:

{cpp_code}

However, the C++ code failed verification.

The compiler/runtime error was:

{error}

Fix the C++ code so that:
- It preserves the behavior of the Python code.
- It is complete standalone C++17.
- It includes all required headers.
- It contains a main() function.
- It compiles and runs correctly.

Return only the corrected C++ code.
"""

Simulate error by crafting an incorrect 'response' for simple Hello World python script

In [15]:
bad_python = """
def greet():
    print("Hello World")

greet()
"""

bad_cpp = """
#include <iostream>

int main() {
    std::cout << "Hello World"
    return 0;
}
"""

success, error = compile_and_run(bad_cpp)

print("Success:", success)
print("Compiler error:")
print(error)

Success: False
Compiler error:
/tmp/tmpjb1_pivq/program.cpp: In function ‘int main()’:
/tmp/tmpjb1_pivq/program.cpp:5:31: error: expected ‘;’ before ‘return’
    5 |     std::cout << "Hello World"
      |                               ^
      |                               ;
    6 |     return 0;
      |     ~~~~~~                     



Test faulty code correction

In [16]:
# Use the bad C++ and the compiler error from the previous test

fix_prompt = make_fix_prompt(
    bad_python,
    bad_cpp,
    error
)

messages = [
    {
        "role": "system",
        "content": "You are an expert Python-to-C++ translator."
    },
    {
        "role": "user",
        "content": fix_prompt
    }
]

response_fixed = generate_response(messages)

print(response_fixed)

Certainly! Below is the corrected C++17 code that replicates the behavior of the given Python code:

```cpp
#include <iostream>

int main() {
    std::cout << "Hello World" << std::endl;
    return 0;
}
```

### Key Changes:
1. **`std::endl`**: Added `std::endl` to ensure the output is followed by a newline character, similar to the `print()` function in Python.
2. **Semicolon**: Added a semicolon at the end of the `std::cout` statement to properly terminate the statement.
3. **Main Function**: Ensured the `main()` function is complete and returns an integer value (`0` in this case).


### 9. Retry Loop
Update `translate_and_verify` to use `make_fix_prompt` and manage only the iterative retry loop

In [17]:
def translate_and_verify(python_code, max_retries=3):

    # Initial translation prompt
    prompt = f"""
Translate the following Python code into standalone C++17.

Requirements:
- Preserve the behavior of the Python code.
- Produce complete compilable C++17 code.
- Include all required headers.
- Include a main() function.
- Return only the C++ code.

Python code:

{python_code}
"""

    messages = [
        {
            "role": "system",
            "content": "You are an expert Python-to-C++ translator."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    # Try up to max_retries times
    for attempt in range(max_retries):

        print(f"\nAttempt {attempt + 1}/{max_retries}")

        # 1. Ask Qwen for C++
        response = generate_response(messages)

        # 2. Extract clean C++
        cpp_code = extract_cpp(response)

        # 3. Compile and execute it
        success, result = compile_and_run(cpp_code)

        # 4. If it works, we're done
        if success:
            print("✓ Compilation and execution successful!")
            return cpp_code

        # 5. If it failed, show the error
        print("✗ Verification failed.")
        print(result)

        # 6. Give the failed code + error back to Qwen
        fix_prompt = make_fix_prompt(
            python_code,
            cpp_code,
            result
        )

        messages = [
            {
                "role": "system",
                "content": "You are an expert Python-to-C++ translator."
            },
            {
                "role": "user",
                "content": fix_prompt
            }
        ]

    # All attempts failed
    raise RuntimeError(
        f"C++ translation failed after {max_retries} attempts."
    )

Test completed pipeline

In [18]:
final_cpp = translate_and_verify(sample_python, max_retries=3)

print("\nFinal working C++:")
print(final_cpp)


Attempt 1/3
✓ Compilation and execution successful!

Final working C++:
#include <iostream>
#include <vector>

std::vector<int> merge_sort(const std::vector<int>& arr) {
    if (arr.size() <= 1) {
        return arr;
    }

    size_t mid = arr.size() / 2;

    std::vector<int> left = merge_sort(std::vector<int>(arr.begin(), arr.begin() + mid));
    std::vector<int> right = merge_sort(std::vector<int>(arr.begin() + mid, arr.end()));

    std::vector<int> result;
    size_t i = 0, j = 0;

    while (i < left.size() && j < right.size()) {
        if (left[i] <= right[j]) {
            result.push_back(left[i]);
            ++i;
        } else {
            result.push_back(right[j]);
            ++j;
        }
    }

    result.insert(result.end(), left.begin() + i, left.end());
    result.insert(result.end(), right.begin() + j, right.end());

    return result;
}

int main() {
    std::vector<int> arr = {7, 2, 9, 1, 5, 3};
    std::vector<int> sorted_arr = merge_sort(arr);

    std::co

### 10. Main Entry Point and Final Demonstration

In [19]:
if __name__ == "__main__":

    print("=== Python → C++ Translation Agent ===")

    final_cpp = translate_and_verify(
        sample_python,
        max_retries=3
    )

    print("\n=== Final Verified C++ ===")
    print(final_cpp)

=== Python → C++ Translation Agent ===

Attempt 1/3
✓ Compilation and execution successful!

=== Final Verified C++ ===
#include <iostream>
#include <vector>

std::vector<int> merge_sort(const std::vector<int>& arr) {
    if (arr.size() <= 1) {
        return arr;
    }

    size_t mid = arr.size() / 2;

    std::vector<int> left = merge_sort(std::vector<int>(arr.begin(), arr.begin() + mid));
    std::vector<int> right = merge_sort(std::vector<int>(arr.begin() + mid, arr.end()));

    std::vector<int> result;
    size_t i = 0, j = 0;

    while (i < left.size() && j < right.size()) {
        if (left[i] <= right[j]) {
            result.push_back(left[i]);
            ++i;
        } else {
            result.push_back(right[j]);
            ++j;
        }
    }

    result.insert(result.end(), left.begin() + i, left.end());
    result.insert(result.end(), right.begin() + j, right.end());

    return result;
}

int main() {
    std::vector<int> arr = {7, 2, 9, 1, 5, 3};
    std::vector<